# 02 — Document Loading & Chunking

This notebook explores how to load documents and split them into chunks for embedding.

**What you'll learn:**
- Loading text files with LangChain document loaders
- Splitting documents with RecursiveCharacterTextSplitter
- How chunk size and overlap affect the output
- Combining chunking with semantic search from Lesson 2

## 1. Load documents from the sample data folder

In [ ]:
!pip install langchain_community

In [1]:
import sys, os
os.chdir(os.path.join(os.path.dirname(os.path.abspath('.')), ''))
# Ensure the src package is importable
sys.path.insert(0, 'src')

from rag_pipeline.chunking import load_directory, chunk_documents

docs = load_directory('data/sample/')

print(f'Loaded {len(docs)} documents\n')
for doc in docs:
    print(f"  {doc.metadata['source']}")
    print(f"    {len(doc.page_content):,} characters")
    print(f"    Preview: {doc.page_content[:80]}...\n")

/Users/nischal/Desktop/Vault/03_Projects/RAG/project/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Loaded 2 documents

  data/sample/company_handbook.txt
    7,992 characters
    Preview: ACME Corporation Employee Handbook
Last Updated: January 2026

Chapter 1: Welcom...

  data/sample/engineering_wiki.txt
    6,478 characters
    Preview: ACME Engineering Wiki
Last Updated: March 2026

=== Development Environment Setu...



## 2. Chunk with the default settings (500 chars, 50 overlap)

In [2]:
chunks = chunk_documents(docs, chunk_size=500, chunk_overlap=50)

print(f'Split {len(docs)} documents into {len(chunks)} chunks\n')
print('--- First 3 chunks ---\n')
for i, chunk in enumerate(chunks[:3]):
    print(f'Chunk {i} ({len(chunk.page_content)} chars):')
    print(f'  Source: {chunk.metadata["source"]}')
    print(f'  Index:  {chunk.metadata["chunk_index"]} of {chunk.metadata["chunk_total"]}')
    print(f'  Text:   {chunk.page_content[:120]}...\n')

Split 2 documents into 40 chunks

--- First 3 chunks ---

Chunk 0 (89 chars):
  Source: data/sample/company_handbook.txt
  Index:  0 of 23
  Text:   ACME Corporation Employee Handbook
Last Updated: January 2026

Chapter 1: Welcome to ACME...

Chunk 1 (446 chars):
  Source: data/sample/company_handbook.txt
  Index:  1 of 23
  Text:   Chapter 1: Welcome to ACME

Welcome to ACME Corporation! We are glad you have chosen to join our team. This handbook is ...

Chunk 2 (398 chars):
  Source: data/sample/company_handbook.txt
  Index:  2 of 23
  Text:   No employee handbook can anticipate every circumstance or question about policy. As ACME continues to grow, the need may...



## 3. Inspect chunk size distribution

In [3]:
import statistics

sizes = [len(c.page_content) for c in chunks]

print(f'Chunk count: {len(sizes)}')
print(f'Mean size:   {statistics.mean(sizes):.0f} chars')
print(f'Median size: {statistics.median(sizes):.0f} chars')
print(f'Min size:    {min(sizes)} chars')
print(f'Max size:    {max(sizes)} chars')
print(f'Std dev:     {statistics.stdev(sizes):.0f} chars')

# Simple text histogram
print('\nSize distribution:')
bins = [0, 100, 200, 300, 400, 500, 600]
for i in range(len(bins) - 1):
    count = sum(1 for s in sizes if bins[i] <= s < bins[i+1])
    bar = '█' * count
    print(f'  {bins[i]:>4}-{bins[i+1]:<4} chars: {bar} ({count})')

Chunk count: 40
Mean size:   369 chars
Median size: 364 chars
Min size:    89 chars
Max size:    494 chars
Std dev:     97 chars

Size distribution:
     0-100  chars: █ (1)
   100-200  chars:  (0)
   200-300  chars: █████████ (9)
   300-400  chars: █████████████ (13)
   400-500  chars: █████████████████ (17)
   500-600  chars:  (0)


## 4. Compare different chunk sizes

In [4]:
print(f'{"Size":>6}  {"Overlap":>7}  {"Chunks":>6}  {"Avg chars":>9}  {"Min":>4}  {"Max":>4}')
print('-' * 50)

for size in [250, 500, 750, 1000]:
    overlap = int(size * 0.1)
    result = chunk_documents(docs, chunk_size=size, chunk_overlap=overlap)
    s = [len(c.page_content) for c in result]
    print(f'{size:>6}  {overlap:>7}  {len(result):>6}  {statistics.mean(s):>9.0f}  {min(s):>4}  {max(s):>4}')

  Size  Overlap  Chunks  Avg chars   Min   Max
--------------------------------------------------
   250       25      98        147    19   244
   500       50      40        369    89   494
   750       75      24        615   274   748
  1000      100      18        825   255   988


## 5. Verify overlap between consecutive chunks

In [ ]:
# Get chunks from a single document to check overlap
source = chunks[0].metadata['source']
same_source = [c for c in chunks if c.metadata['source'] == source]

print(f'Checking overlap for: {source}')
print(f'Chunks from this document: {len(same_source)}\n')

# Check first pair
c1 = same_source[0].page_content
c2 = same_source[1].page_content

print('End of chunk 0 (last 80 chars):')
print(f'  "{c1[-80:]}"')
print(f'\nStart of chunk 1 (first 80 chars):')
print(f'  "{c2[:80]}"')

# Find the actual overlap
for overlap_len in range(min(len(c1), len(c2)), 0, -1):
    if c1.endswith(c2[:overlap_len]):
        print(f'\nActual overlap: {overlap_len} characters')
        print(f'Overlapping text: "{c2[:overlap_len]}"')
        break
else:
    print('\nNo overlap found (chunks may have been split at a paragraph boundary)')

## 6. Combining chunking with semantic search

Now let's connect Lesson 2 (embeddings) with Lesson 3 (chunking).
We'll chunk our documents, then search them with natural language queries.

In [ ]:
from rag_pipeline.embeddings import load_model, rank_by_similarity

# Load the embedding model
model = load_model()

# Use our 500-char chunks
chunk_texts = [c.page_content for c in chunks]
chunk_metadata = [c.metadata for c in chunks]

# Search!
queries = [
    'What is the remote work policy?',
    'How do I deploy to production?',
    'How much PTO do I get?',
    'What is the code review process?',
]

for query in queries:
    print(f'\n{"=" * 60}')
    print(f'Query: "{query}"\n')
    results = rank_by_similarity(query, chunk_texts, model, top_k=2)
    for rank, (text, score) in enumerate(results, 1):
        # Find the metadata for this chunk
        idx = chunk_texts.index(text)
        meta = chunk_metadata[idx]
        source_name = meta['source'].split('/')[-1]
        print(f'  #{rank} (score: {score:.3f}) [{source_name}, chunk {meta["chunk_index"]}]')
        print(f'  {text[:150]}...\n')

## Key Takeaways

1. **Document loaders** extract text + metadata from files (`.txt`, `.pdf`, etc.)
2. **RecursiveCharacterTextSplitter** is the industry-standard chunking strategy
3. **Chunk size = 500, overlap = 50** is a solid starting point
4. **Metadata** follows each chunk so you can cite sources in answers
5. **Chunking + embedding + similarity search** = the retrieval half of RAG

**Next:** We'll store these embeddings in a vector database (ChromaDB) so we don't
have to re-embed every time we search.